In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from scenarios.price_scenarios import build_zonal_price_matrix, zone_annual_cf, CONTRACT_START

own_zone = "nord"
reference_zone = "sicilia"

price_own_zone = build_zonal_price_matrix(own_zone)
price_reference_zone = build_zonal_price_matrix(reference_zone)
price_own_zone.head()

price_scenario Delayed transition                                      \
weather_year                 1991        1992        1993        1994   
contract_year                                                           
2026                   103.875909  104.316244  104.055878  104.264543   
2027                   101.454880  101.895215  101.634849  101.843513   
2028                    99.033851   99.474185   99.213819   99.422484   
2029                    96.612821   97.053156   96.792790   97.001455   
2030                    94.191792   94.632127   94.371761   94.580426   

price_scenario                                                              \
weather_year          1995        1996        1997        1998        1999   
contract_year                                                                
2026            103.783605  104.723265  103.665408  103.587186  104.067303   
2027            101.362575  102.302236  101.244378  101.166157  101.646274   
2028             98.941546   99.881206   98.823349   98.745128   99.225244   
2029             96.520517   97.460177   96.402320   96.324099   96.804215   
2030             94.099488   95.039148   93.981291   93.903069   94.383186   

price_scenario              ... Net Zero 2050                          \
weather_year          2000  ...          2014        2015        2016   
contract_year               ...                                         
2026            104.040144  ...    110.643072  109.916631  110.109090   
2027            101.619114  ...    111.375168  110.648728  110.841187   
2028             99.198085  ...    112.107264  111.380824  111.573283   
2029             96.777056  ...    112.839361  112.112920  112.305379   
2030             94.356027  ...    113.571457  112.845017  113.037476   

price_scenario                                                              \
weather_year          2017        2018        2019        2020        2021   
contract_year                                                                
2026            109.508766  110.246683  109.840647  109.721979  109.802884   
2027            110.240862  110.978779  110.572743  110.454075  110.534981   
2028            110.972959  111.710875  111.304839  111.186172  111.267077   
2029            111.705055  112.442972  112.036936  111.918268  111.999173   
2030            112.437151  113.175068  112.769032  112.650364  112.731270   

price_scenario                          
weather_year          2022        2023  
contract_year                           
2026            109.546311  109.692967  
2027            110.278407  110.425063  
2028            111.010504  111.157160  
2029            111.742600  111.889256  
2030            112.474696  112.621352  

[5 rows x 99 columns]

In [2]:
annual_solar_cf = zone_annual_cf("solar", own_zone)
annual_solar_cf.describe()

count    33.000000
mean      0.181007
std       0.006687
min       0.167394
25%       0.176553
50%       0.181215
75%       0.186130
max       0.190684
dtype: float64

In [3]:
STRIKE_PRICE_SOLAR = 56.83
HOURS_PER_YEAR = 8760

pap_payment_per_mw = STRIKE_PRICE_SOLAR * annual_solar_cf * HOURS_PER_YEAR
pap_payment_per_mw.describe()

count       33.000000
mean     90110.949264
std       3329.118366
min      83333.888435
25%      87893.422921
50%      90214.482238
75%      92661.070039
max      94928.422168
dtype: float64

In [4]:
from scenarios.load_profile import load_profile_for_archetype

annual_kwh = 20_000_000
load = load_profile_for_archetype("chemicals", annual_kwh)
load.sum() / 1000

np.float64(20000.0)

In [5]:
annual_load_mwh = load.sum() / 1000
CONTRACTED_MW = 5

residual_mwh = (annual_load_mwh - annual_solar_cf * HOURS_PER_YEAR * CONTRACTED_MW).clip(lower=0)
residual_mwh.describe()

count       33.000000
mean     12071.885513
std        292.901493
min      11648.036058
25%      11847.521552
50%      12062.776506
75%      12266.987250
max      12668.142844
dtype: float64

In [6]:
payment_total = pap_payment_per_mw * CONTRACTED_MW

columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_price = price_own_zone[(scenario, weather_year)]
    columns[(scenario, weather_year)] = payment_total[weather_year] + residual_mwh[weather_year] * spot_price

pap_solar_cost_matrix = pd.DataFrame(columns)
pap_solar_cost_matrix.columns.names = ["price_scenario", "weather_year"]
pap_solar_cost_matrix.index.name = "contract_year"
pap_solar_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.697635e+06  1.721049e+06  1.707469e+06  1.718479e+06   
2027                 1.668764e+06  1.691252e+06  1.678203e+06  1.688780e+06   
2028                 1.639892e+06  1.661454e+06  1.648938e+06  1.659081e+06   
2029                 1.611021e+06  1.631657e+06  1.619672e+06  1.629383e+06   
2030                 1.582149e+06  1.601859e+06  1.590407e+06  1.599684e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.692884e+06  1.743319e+06  1.686948e+06  1.682637e+06   
2027            1.664201e+06  1.712649e+06  1.658499e+06  1.654364e+06   
2028            1.635517e+06  1.681979e+06  1.630051e+06  1.626091e+06   
2029            1.606834e+06  1.651309e+06  1.601602e+06  1.597818e+06   
2030            1.578151e+06  1.620639e+06  1.573153e+06  1.569544e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.708147e+06  1.706214e+06  ...  1.815988e+06  1.773613e+06   
2027            1.678854e+06  1.677003e+06  ...  1.825231e+06  1.782398e+06   
2028            1.649561e+06  1.647792e+06  ...  1.834474e+06  1.791182e+06   
2029            1.620267e+06  1.618581e+06  ...  1.843716e+06  1.799967e+06   
2030            1.590974e+06  1.589370e+06  ...  1.852959e+06  1.808752e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.784925e+06  1.750204e+06  1.792926e+06  1.769508e+06   
2027            1.793834e+06  1.758732e+06  1.801921e+06  1.778249e+06   
2028            1.802742e+06  1.767259e+06  1.810917e+06  1.786989e+06   
2029            1.811651e+06  1.775787e+06  1.819912e+06  1.795730e+06   
2030            1.820559e+06  1.784314e+06  1.828907e+06  1.804471e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.762305e+06  1.766904e+06  1.751863e+06  1.760555e+06  
2027            1.770965e+06  1.775615e+06  1.760408e+06  1.769196e+06  
2028            1.779626e+06  1.784326e+06  1.768952e+06  1.777837e+06  
2029            1.788287e+06  1.793037e+06  1.777497e+06  1.786478e+06  
2030            1.796947e+06  1.801748e+06  1.786041e+06  1.795119e+06  

[5 rows x 99 columns]

In [7]:
STRIKE_PRICE_WIND = 72.85

annual_wind_cf = zone_annual_cf("wind", own_zone)
wind_payment_per_mw = STRIKE_PRICE_WIND * annual_wind_cf * HOURS_PER_YEAR
residual_wind_mwh = (annual_load_mwh - annual_wind_cf * HOURS_PER_YEAR * CONTRACTED_MW).clip(lower=0)
residual_wind_mwh.describe()

count       33.000000
mean     16845.381647
std        298.416088
min      16324.612254
25%      16706.626247
50%      16862.255550
75%      17033.505189
max      17414.223108
dtype: float64

In [8]:
wind_payment_total = wind_payment_per_mw * CONTRACTED_MW

wind_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_price = price_own_zone[(scenario, weather_year)]
    wind_columns[(scenario, weather_year)] = wind_payment_total[weather_year] + residual_wind_mwh[weather_year] * spot_price

pap_wind_cost_matrix = pd.DataFrame(wind_columns)
pap_wind_cost_matrix.columns.names = ["price_scenario", "weather_year"]
pap_wind_cost_matrix.index.name = "contract_year"
pap_wind_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.987088e+06  1.995895e+06  1.973029e+06  1.983462e+06   
2027                 1.945724e+06  1.954432e+06  1.932995e+06  1.942889e+06   
2028                 1.904360e+06  1.912969e+06  1.892960e+06  1.902316e+06   
2029                 1.862996e+06  1.871506e+06  1.852925e+06  1.861743e+06   
2030                 1.821632e+06  1.830043e+06  1.812890e+06  1.821170e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.978658e+06  1.984973e+06  1.960584e+06  1.971468e+06   
2027            1.937831e+06  1.944869e+06  1.921020e+06  1.930946e+06   
2028            1.897003e+06  1.904765e+06  1.881455e+06  1.890423e+06   
2029            1.856175e+06  1.864662e+06  1.841891e+06  1.849901e+06   
2030            1.815347e+06  1.824558e+06  1.802327e+06  1.809379e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.969130e+06  1.997010e+06  ...  2.115137e+06  2.082027e+06   
2027            1.929413e+06  1.955093e+06  ...  2.127886e+06  2.094372e+06   
2028            1.889695e+06  1.913177e+06  ...  2.140635e+06  2.106717e+06   
2029            1.849977e+06  1.871260e+06  ...  2.153384e+06  2.119061e+06   
2030            1.810259e+06  1.829344e+06  ...  2.166133e+06  2.131406e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            2.080040e+06  2.064695e+06  2.086352e+06  2.061791e+06   
2027            2.092282e+06  2.076831e+06  2.098672e+06  2.073760e+06   
2028            2.104524e+06  2.088967e+06  2.110993e+06  2.085730e+06   
2029            2.116766e+06  2.101103e+06  2.123313e+06  2.097700e+06   
2030            2.129008e+06  2.113239e+06  2.135634e+06  2.109669e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            2.079867e+06  2.086437e+06  2.095772e+06  2.083849e+06  
2027            2.092235e+06  2.098907e+06  2.108516e+06  2.096305e+06  
2028            2.104602e+06  2.111377e+06  2.121260e+06  2.108761e+06  
2029            2.116969e+06  2.123848e+06  2.134003e+06  2.121217e+06  
2030            2.129336e+06  2.136318e+06  2.146747e+06  2.133673e+06  

[5 rows x 99 columns]

In [9]:
BASELOAD_DISCOUNT = 0.14
CONTRACTED_MW_BASELOAD = 2

reference_baseload_price = price_own_zone.mean().mean()
strike_baseload = reference_baseload_price * (1 - BASELOAD_DISCOUNT)
strike_baseload

np.float64(82.94876862842975)

In [10]:
baseload_payment = strike_baseload * CONTRACTED_MW_BASELOAD * HOURS_PER_YEAR
residual_baseload_mwh = annual_load_mwh - CONTRACTED_MW_BASELOAD * HOURS_PER_YEAR

baseload_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_price = price_own_zone[(scenario, weather_year)]
    baseload_columns[(scenario, weather_year)] = baseload_payment + residual_baseload_mwh * spot_price

baseload_cost_matrix = pd.DataFrame(baseload_columns)
baseload_cost_matrix.columns.names = ["price_scenario", "weather_year"]
baseload_cost_matrix.index.name = "contract_year"
baseload_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.710875e+06  1.711967e+06  1.711321e+06  1.711838e+06   
2027                 1.704871e+06  1.705963e+06  1.705317e+06  1.705834e+06   
2028                 1.698866e+06  1.699958e+06  1.699313e+06  1.699830e+06   
2029                 1.692862e+06  1.693954e+06  1.693309e+06  1.693826e+06   
2030                 1.686858e+06  1.687950e+06  1.687304e+06  1.687822e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.710646e+06  1.712976e+06  1.710353e+06  1.710159e+06   
2027            1.704642e+06  1.706972e+06  1.704348e+06  1.704154e+06   
2028            1.698637e+06  1.700968e+06  1.698344e+06  1.698150e+06   
2029            1.692633e+06  1.694964e+06  1.692340e+06  1.692146e+06   
2030            1.686629e+06  1.688960e+06  1.686336e+06  1.686142e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.711349e+06  1.711282e+06  ...  1.727657e+06  1.725856e+06   
2027            1.705345e+06  1.705278e+06  ...  1.729473e+06  1.727671e+06   
2028            1.699341e+06  1.699274e+06  ...  1.731288e+06  1.729487e+06   
2029            1.693337e+06  1.693270e+06  ...  1.733104e+06  1.731302e+06   
2030            1.687333e+06  1.687265e+06  ...  1.734920e+06  1.733118e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.726333e+06  1.724844e+06  1.726674e+06  1.725667e+06   
2027            1.728149e+06  1.726660e+06  1.728490e+06  1.727483e+06   
2028            1.729964e+06  1.728475e+06  1.730305e+06  1.729298e+06   
2029            1.731780e+06  1.730291e+06  1.732121e+06  1.731114e+06   
2030            1.733595e+06  1.732107e+06  1.733937e+06  1.732930e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.725373e+06  1.725574e+06  1.724937e+06  1.725301e+06  
2027            1.727189e+06  1.727389e+06  1.726753e+06  1.727117e+06  
2028            1.729004e+06  1.729205e+06  1.728568e+06  1.728932e+06  
2029            1.730820e+06  1.731020e+06  1.730384e+06  1.730748e+06  
2030            1.732635e+06  1.732836e+06  1.732200e+06  1.732563e+06  

[5 rows x 99 columns]

In [11]:
SLEEVING_MARGIN = 3.0

sleeved_solar_mwh = annual_solar_cf * HOURS_PER_YEAR * CONTRACTED_MW
sleeving_fee_total = SLEEVING_MARGIN * sleeved_solar_mwh
sleeving_fee_total.describe()

count       33.000000
mean     23784.343462
std        878.704478
min      21995.571468
25%      23199.038251
50%      23811.670483
75%      24457.435344
max      25055.891827
dtype: float64

In [12]:
sleeved_columns = {}
for scenario, weather_year in price_own_zone.columns:
    sleeved_columns[(scenario, weather_year)] = (
        pap_solar_cost_matrix[(scenario, weather_year)] + sleeving_fee_total[weather_year]
    )

sleeved_cost_matrix = pd.DataFrame(sleeved_columns)
sleeved_cost_matrix.columns.names = ["price_scenario", "weather_year"]
sleeved_cost_matrix.index.name = "contract_year"
sleeved_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.721859e+06  1.744126e+06  1.731205e+06  1.741678e+06   
2027                 1.692988e+06  1.714328e+06  1.701939e+06  1.711979e+06   
2028                 1.664116e+06  1.684531e+06  1.672674e+06  1.682280e+06   
2029                 1.635245e+06  1.654734e+06  1.643408e+06  1.652582e+06   
2030                 1.606374e+06  1.624936e+06  1.614143e+06  1.622883e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.717341e+06  1.765314e+06  1.711696e+06  1.707603e+06   
2027            1.688658e+06  1.734644e+06  1.683247e+06  1.679329e+06   
2028            1.659975e+06  1.703974e+06  1.654799e+06  1.651056e+06   
2029            1.631292e+06  1.673304e+06  1.626350e+06  1.622783e+06   
2030            1.602608e+06  1.642635e+06  1.597901e+06  1.594510e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.731848e+06  1.730017e+06  ...  1.838113e+06  1.797614e+06   
2027            1.702555e+06  1.700806e+06  ...  1.847356e+06  1.806399e+06   
2028            1.673262e+06  1.671596e+06  ...  1.856599e+06  1.815184e+06   
2029            1.643969e+06  1.642385e+06  ...  1.865842e+06  1.823969e+06   
2030            1.614676e+06  1.613174e+06  ...  1.875084e+06  1.832753e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.808420e+06  1.775260e+06  1.816065e+06  1.793690e+06   
2027            1.817328e+06  1.783788e+06  1.825061e+06  1.802431e+06   
2028            1.826237e+06  1.792315e+06  1.834056e+06  1.811172e+06   
2029            1.835145e+06  1.800843e+06  1.843051e+06  1.819912e+06   
2030            1.844054e+06  1.809370e+06  1.852046e+06  1.828653e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.786815e+06  1.791208e+06  1.776850e+06  1.785145e+06  
2027            1.795476e+06  1.799919e+06  1.785394e+06  1.793786e+06  
2028            1.804136e+06  1.808630e+06  1.793939e+06  1.802427e+06  
2029            1.812797e+06  1.817341e+06  1.802483e+06  1.811068e+06  
2030            1.821458e+06  1.826052e+06  1.811028e+06  1.819709e+06  

[5 rows x 99 columns]

In [13]:
basis_risk = price_own_zone - price_reference_zone
basis_risk.loc[CONTRACT_START].describe()

count    9.900000e+01
mean     2.153160e-15
std      4.270334e-01
min     -6.005946e-01
25%     -4.115476e-01
50%     -1.651287e-02
75%      1.980173e-01
max      1.023159e+00
Name: 2026, dtype: float64

In [14]:
CONTRACTED_MW_VPPA = 5

wind_cf_reference = zone_annual_cf("wind", reference_zone)

vppa_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_own = price_own_zone[(scenario, weather_year)]
    spot_reference = price_reference_zone[(scenario, weather_year)]
    settlement = (STRIKE_PRICE_WIND - spot_reference) * wind_cf_reference[weather_year] * HOURS_PER_YEAR * CONTRACTED_MW_VPPA
    vppa_columns[(scenario, weather_year)] = annual_load_mwh * spot_own + settlement

vppa_cost_matrix = pd.DataFrame(vppa_columns)
vppa_cost_matrix.columns.names = ["price_scenario", "weather_year"]
vppa_cost_matrix.index.name = "contract_year"
vppa_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.767057e+06  1.774301e+06  1.761999e+06  1.769014e+06   
2027                 1.742642e+06  1.749996e+06  1.738323e+06  1.745109e+06   
2028                 1.718227e+06  1.725692e+06  1.714648e+06  1.721204e+06   
2029                 1.693812e+06  1.701387e+06  1.690972e+06  1.697299e+06   
2030                 1.669396e+06  1.677082e+06  1.667296e+06  1.673394e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.734897e+06  1.687642e+06  1.759600e+06  1.744869e+06   
2027            1.712990e+06  1.671148e+06  1.735451e+06  1.721842e+06   
2028            1.691083e+06  1.654654e+06  1.711301e+06  1.698815e+06   
2029            1.669177e+06  1.638159e+06  1.687151e+06  1.675789e+06   
2030            1.647270e+06  1.621665e+06  1.663001e+06  1.652762e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.751147e+06  1.752613e+06  ...  1.807354e+06  1.828984e+06   
2027            1.728405e+06  1.729695e+06  ...  1.813989e+06  1.836369e+06   
2028            1.705663e+06  1.706777e+06  ...  1.820624e+06  1.843755e+06   
2029            1.682922e+06  1.683859e+06  ...  1.827260e+06  1.851140e+06   
2030            1.660180e+06  1.660941e+06  ...  1.833895e+06  1.858526e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.786454e+06  1.815119e+06  1.804092e+06  1.768398e+06   
2027            1.792884e+06  1.822372e+06  1.810836e+06  1.774559e+06   
2028            1.799313e+06  1.829624e+06  1.817580e+06  1.780721e+06   
2029            1.805743e+06  1.836876e+06  1.824325e+06  1.786882e+06   
2030            1.812173e+06  1.844129e+06  1.831069e+06  1.793043e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.859030e+06  1.781493e+06  1.843056e+06  1.849572e+06  
2027            1.867100e+06  1.787949e+06  1.850869e+06  1.857462e+06  
2028            1.875171e+06  1.794405e+06  1.858683e+06  1.865352e+06  
2029            1.883241e+06  1.800861e+06  1.866497e+06  1.873242e+06  
2030            1.891311e+06  1.807317e+06  1.874310e+06  1.881131e+06  

[5 rows x 99 columns]

In [15]:
spot_only_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_only_columns[(scenario, weather_year)] = annual_load_mwh * price_own_zone[(scenario, weather_year)]

spot_only_cost_matrix = pd.DataFrame(spot_only_columns)
spot_only_cost_matrix.columns.names = ["price_scenario", "weather_year"]
spot_only_cost_matrix.index.name = "contract_year"
spot_only_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 2.077518e+06  2.086325e+06  2.081118e+06  2.085291e+06   
2027                 2.029098e+06  2.037904e+06  2.032697e+06  2.036870e+06   
2028                 1.980677e+06  1.989484e+06  1.984276e+06  1.988450e+06   
2029                 1.932256e+06  1.941063e+06  1.935856e+06  1.940029e+06   
2030                 1.883836e+06  1.892643e+06  1.887435e+06  1.891609e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            2.075672e+06  2.094465e+06  2.073308e+06  2.071744e+06   
2027            2.027252e+06  2.046045e+06  2.024888e+06  2.023323e+06   
2028            1.978831e+06  1.997624e+06  1.976467e+06  1.974903e+06   
2029            1.930410e+06  1.949204e+06  1.928046e+06  1.926482e+06   
2030            1.881990e+06  1.900783e+06  1.879626e+06  1.878061e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            2.081346e+06  2.080803e+06  ...  2.212861e+06  2.198333e+06   
2027            2.032925e+06  2.032382e+06  ...  2.227503e+06  2.212975e+06   
2028            1.984505e+06  1.983962e+06  ...  2.242145e+06  2.227616e+06   
2029            1.936084e+06  1.935541e+06  ...  2.256787e+06  2.242258e+06   
2030            1.887664e+06  1.887121e+06  ...  2.271429e+06  2.256900e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            2.202182e+06  2.190175e+06  2.204934e+06  2.196813e+06   
2027            2.216824e+06  2.204817e+06  2.219576e+06  2.211455e+06   
2028            2.231466e+06  2.219459e+06  2.234218e+06  2.226097e+06   
2029            2.246108e+06  2.234101e+06  2.248859e+06  2.240739e+06   
2030            2.260750e+06  2.248743e+06  2.263501e+06  2.255381e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            2.194440e+06  2.196058e+06  2.190926e+06  2.193859e+06  
2027            2.209082e+06  2.210700e+06  2.205568e+06  2.208501e+06  
2028            2.223723e+06  2.225342e+06  2.220210e+06  2.223143e+06  
2029            2.238365e+06  2.239983e+06  2.234852e+06  2.237785e+06  
2030            2.253007e+06  2.254625e+06  2.249494e+06  2.252427e+06  

[5 rows x 99 columns]